# 05 · 3-axis CNC stage — manual control

Manual jog panel for a G-code CNC / XYZ stage (Marlin/GRBL/RepRap) over USB,
used to position the EMFI probe over the target (ChipShover-style). The
controller and the ipywidgets panel live in the package (`faultycat.cnc`); this
notebook just wires them up.

> ⚠️ Know your stage's mechanical limits. Software endstops are turned off
> (`M211 S0`) so negative coordinates work, so there is no software floor near
> the ends of travel — use small steps there. The STOP button sends `M112`
> (emergency stop); the board must be reset afterwards.

## 1 · Connect

Set the port/baud for your stage and open it. On `permission denied`, run
`sudo chmod a+rw /dev/ttyUSB0` (or add your user to the `dialout` group).

Not sure which port? `from serial.tools import list_ports` then
`[print(p.device, p.description) for p in list_ports.comports()]`.

In [ ]:
import faultycat as fc

PORT, BAUD = "/dev/ttyUSB0", 115200   # e.g. /dev/ttyUSB0, /dev/ttyACM0, COM3
stage = fc.CncStage(PORT, BAUD)
stage

## 2 · Manual control panel

| Control | Action |
| ------- | ------ |
| Crosshair X/Y, Z+/Z- | Relative jog by the current step |
| **Step (mm)** / **Speed XY, Z** | Jog distance and feedrate (mm/min) |
| ⌂ **home** / **Home X·Y·Z** | `G28` all axes or per axis |
| 📍 **Position** | Read `M114`, update the readout |
| **Motors off** / 🔌 **Reconnect** / 🛑 **STOP** | `M84` / reopen port / `M112` |
| **Allow negatives** | `M211 S0/S1` software endstops |
| **Set origin here** / **Go to origin** | `G92 X0 Y0 Z0` / `G1 X0 Y0 Z0` |
| **G-code** | Send a manual line |

The CNC boots assuming it is at `0,0,0` wherever the head happens to be (no
homing, no saved position). Workflow: jog the probe over the chip, click **Set
origin here**, then move in +/- around that point.

In [ ]:
fc.cnc_panel(stage)

## Diagnostics (run only if a button errors)

Opens the port raw and dumps pyserial's state plus a full traceback on any read
failure. Paste its whole output.

In [ ]:
fc.cnc_diagnostics(PORT, BAUD)

## Close

Release the serial port when done (or before reconnecting elsewhere).

In [ ]:
stage.close()

## Next steps

Manual control only, for now. The base to build on:

- **XY spatial sweep** of the EMFI probe: step over a grid around the work
  origin, fire the FaultyCat glitch at each point and classify the result into a
  **spatial fault map** (X, Y, hit).
- Combine stage position with `fc.GlitchController` (delay/width) in one
  campaign.
- Soft travel limits and a safe start sequence.